# 10 · TTF Spread Deep Dive

Calendar spread analysis: contango/backwardation dynamics, roll yield and seasonality.

**Data sources:**
- TTF forward curve: `data/raw/ttf_curve.csv`
- EU storage: `data/processed/eu_aggregate_full.parquet`

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
import calendar
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"✅ Root: {_c}"); break

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
ANALYSIS_DATE = None          # None = last data point
START_DATE    = "2020-01-01"
# ═══════════════════════════════════════════════════════════════════

from src.analysis.spread_analysis import (
    build_spread_matrix, roll_yield, spread_regime,
    contango_duration, spread_seasonality
)

ttf_path     = Path("data/raw/ttf_curve.csv")
storage_path = Path("data/processed/eu_aggregate_full.parquet")

ttf_df     = pd.read_csv(ttf_path, index_col=0, parse_dates=True) if ttf_path.exists() else pd.DataFrame()
storage_df = pd.read_parquet(storage_path) if storage_path.exists() else pd.DataFrame()

if not ttf_df.empty:
    ttf_df = ttf_df[ttf_df.index >= START_DATE].sort_index()
if not storage_df.empty:
    storage_df = storage_df[storage_df.index >= START_DATE].sort_index()


# Strip timezone info — TTF CSV is tz-aware (UTC), parquet is tz-naive
for _df in [ttf_df, storage_df]:
    try:
        _df.index = pd.to_datetime(_df.index).tz_localize(None)
    except Exception:
        pass
analysis_date = (
    ANALYSIS_DATE if ANALYSIS_DATE
    else (ttf_df.index[-1].date() if not ttf_df.empty else date.today())
)
print(f"Analysis date: {analysis_date}")
print(f"TTF rows: {len(ttf_df)}, Storage rows: {len(storage_df)}")

## 1 · Full Spread Matrix Heatmap (latest date)

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data. Run notebook 01 / 07 first.")
else:
    sm = build_spread_matrix(ttf_df)
    month_cols = sorted([c for c in ttf_df.columns if c.startswith("M") and c[1:].isdigit()],
                        key=lambda x: int(x[1:]))

    # Build NxN matrix for latest date
    latest = sm.loc[sm.index <= pd.Timestamp(analysis_date)].iloc[-1]
    n = len(month_cols)
    mat = np.full((n, n), np.nan)
    for col in sm.columns:
        parts = col.split("_")
        if len(parts) == 2:
            try:
                i = month_cols.index(parts[0])
                j = month_cols.index(parts[1])
                mat[i, j] = latest.get(col, np.nan)
            except ValueError:
                pass

    fig = go.Figure(go.Heatmap(
        z=mat, x=month_cols, y=month_cols,
        colorscale="RdBu", zmid=0,
        text=np.where(np.isnan(mat), "", np.round(mat, 2).astype(str)),
        texttemplate="%{text}",
        colorbar=dict(title="€/MWh")
    ))
    fig.update_layout(
        title=f"TTF Spread Matrix — {analysis_date} (row M_n minus col M_m)",
        template="plotly_white", height=520,
        xaxis_title="Far month", yaxis_title="Near month"
    )
    fig.show()

## 2 · Roll Yield Over Time

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    ry = roll_yield(ttf_df, holding_days=30)

    fig = go.Figure()
    colors = np.where(ry["roll_yield_pct"] >= 0, "#22c55e", "#ef4444")
    fig.add_trace(go.Bar(x=ry.index, y=ry["roll_yield_pct"],
                         marker_color=colors, name="Roll Yield"))
    fig.add_hline(y=0, line_dash="dot", line_color="black", line_width=1)
    fig.update_layout(template="plotly_white", height=420,
                      title="Annualised Roll Yield — TTF M1 (30d holding period)<br>"
                            "<sup>Green = backwardation (roll profit) | Red = contango (roll cost)</sup>",
                      yaxis_title="% per year", xaxis_title="Date")
    fig.show()
    print(f"Latest roll yield: {ry['roll_yield_pct'].iloc[-1]:.1f}%/year")

## 3 · Contango / Backwardation Regime Timeline

In [ ]:
if ttf_df.empty or storage_df.empty:
    print("⚠️  Need both TTF and storage data.")
else:
    reg_df = spread_regime(ttf_df, storage_df)

    palette = {"contango": "#ef4444", "backwardation": "#22c55e", "flat": "#94a3b8"}
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["M1–M2 Spread (€/MWh)", "EU Fill Rate (%)"],
                        vertical_spacing=0.08, row_heights=[0.6, 0.4])

    for regime, grp in reg_df.groupby("regime"):
        fig.add_trace(go.Scatter(
            x=grp.index, y=grp["spread"], mode="markers",
            marker=dict(color=palette[regime], size=3),
            name=regime
        ), row=1, col=1)

    fig.add_hline(y=0, line_dash="dot", line_color="black", row=1, col=1)
    fig.add_trace(go.Scatter(x=reg_df.index, y=reg_df["fill"],
                             fill="tozeroy", line=dict(color="#6366f1", width=1),
                             fillcolor="rgba(99,102,241,0.15)", name="Fill %",
                             showlegend=False), row=2, col=1)

    fig.update_layout(height=520, template="plotly_white",
                      title="TTF Spread Regime vs EU Storage Fill Rate",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    dur = contango_duration(ttf_df)
    print("Longest streaks:")
    print(dur.nlargest(5, "days")[["start","end","regime","days"]].to_string(index=False))

## 4 · Spread Seasonality (M1-M3, M1-M6, Winter-Summer)

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    seas = spread_seasonality(ttf_df)
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    seas.index = [month_names[i-1] for i in seas.index]

    fig = go.Figure()
    colors_map = {"M1_M3": "#6366f1", "M1_M6": "#f97316"}
    for col in seas.columns:
        fig.add_trace(go.Bar(x=seas.index, y=seas[col],
                             name=col.replace("_", "–"),
                             marker_color=colors_map.get(col, "#94a3b8")))

    # Winter-Summer spread if M1 and M7 available
    if "M1" in ttf_df.columns and "M7" in ttf_df.columns:
        ws = (ttf_df["M1"] - ttf_df["M7"]).groupby(ttf_df.index.month).mean()
        ws.index = [month_names[i-1] for i in ws.index]
        fig.add_trace(go.Bar(x=ws.index, y=ws.values, name="M1–M7 (Win-Sum)",
                             marker_color="#22c55e"))

    fig.add_hline(y=0, line_dash="dot", line_color="black")
    fig.update_layout(template="plotly_white", height=420, barmode="group",
                      title="Average TTF Calendar Spreads by Month",
                      yaxis_title="€/MWh (M_near – M_far)",
                      legend=dict(orientation="h", y=-0.15))
    fig.show()
    print(seas.round(2))

## 5 · Forward Curve Animation

Animated TTF forward curve over time. Y-axis is fixed to the global min/max across all frames so the scale never jumps. X-axis tick labels update per frame to show real delivery months.

In [ ]:
if ttf_df.empty:
    print("⚠️  No TTF data.")
else:
    # ── Tenor columns M1-M12 ────────────────────────────────────────────────
    month_cols = sorted(
        [c for c in ttf_df.columns if c.startswith("M") and c[1:].isdigit()],
        key=lambda x: int(x[1:])
    )[:12]
    x_pos = list(range(1, len(month_cols) + 1))

    # ── Weekly sample dates over last 2 years ────────────────────────────────
    lookback    = ttf_df.index[-1] - pd.DateOffset(years=2)
    window_df   = ttf_df[ttf_df.index >= lookback]
    # Resample to last available day each week
    sample_dates = (
        window_df.resample("W").last().dropna(how="all").index
        .intersection(ttf_df.index)
    )
    if len(sample_dates) < 5:                     # fallback: every 5th row
        sample_dates = window_df.index[::5]

    # ── Helper: real delivery-month tick labels for a given base date ────────
    def delivery_labels(dt):
        labels = []
        for col in month_cols:
            m  = int(col[1:])
            dm = (dt.month + m - 1) % 12 + 1
            yr = dt.year + (dt.month + m - 1) // 12
            labels.append(f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}")
        return labels

    # ── Pre-compute rows and global y-range across ALL frames ────────────────
    frame_data = []   # list of (dt, row_values, tick_labels)
    all_values = []
    for dt in sample_dates:
        row = ttf_df.loc[dt, month_cols].dropna()
        if len(row) < 2:
            continue
        vals   = row.values.tolist()
        labels = delivery_labels(dt)[:len(row)]
        frame_data.append((dt, vals, labels))
        all_values.extend(vals)

    if not frame_data or not all_values:
        print("⚠️  Not enough data for animation.")
    else:
        # Fixed y-range across all frames — axis never jumps
        v_min, v_max = min(all_values), max(all_values)
        pad     = (v_max - v_min) * 0.05
        y_range = [v_min - pad, v_max + pad]

        # ── Build Plotly frames ──────────────────────────────────────────────
        frames = []
        for dt, vals, tick_labels in frame_data:
            pos = list(range(1, len(vals) + 1))
            frames.append(go.Frame(
                data=[go.Scatter(
                    x=pos, y=vals,
                    mode="lines+markers",
                    line=dict(color="#6366f1", width=2.2),
                    marker=dict(size=6, color="#6366f1"),
                    hovertemplate="<b>%{text}</b>: %{y:.2f} €/MWh<extra></extra>",
                    text=tick_labels,
                )],
                # Per-frame layout: update x tick labels to real delivery months
                layout=go.Layout(
                    xaxis=dict(
                        tickmode="array",
                        tickvals=pos,
                        ticktext=tick_labels,
                    )
                ),
                name=str(dt.date()),
            ))

        # ── Initial frame: most recent date ─────────────────────────────────
        init_dt, init_vals, init_labels = frame_data[-1]
        init_pos = list(range(1, len(init_vals) + 1))

        fig = go.Figure(
            data=[go.Scatter(
                x=init_pos, y=init_vals,
                mode="lines+markers",
                line=dict(color="#6366f1", width=2.2),
                marker=dict(size=6, color="#6366f1"),
                hovertemplate="<b>%{text}</b>: %{y:.2f} €/MWh<extra></extra>",
                text=init_labels,
                showlegend=False,
            )],
            frames=frames,
            layout=go.Layout(
                title=dict(text=(
                    "TTF Forward Curve Over Time — Animated<br>"
                    "<sup>Y-axis fixed to global range across all frames. "
                    "X-axis labels show real delivery months per date.</sup>"
                )),
                xaxis=dict(
                    title="Delivery Month",
                    tickmode="array",
                    tickvals=x_pos,
                    ticktext=init_labels,
                    tickangle=-40,
                ),
                yaxis=dict(
                    title="Price (€/MWh)",
                    range=y_range,
                ),
                template="plotly_white",
                height=520,
                updatemenus=[dict(
                    type="buttons",
                    showactive=False,
                    x=0.05, y=1.14, xanchor="left",
                    buttons=[
                        dict(
                            label="▶ Play",
                            method="animate",
                            args=[None, dict(
                                frame=dict(duration=300, redraw=True),
                                fromcurrent=True,
                                transition=dict(duration=0),
                            )],
                        ),
                        dict(
                            label="⏸ Pause",
                            method="animate",
                            args=[[None], dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                            )],
                        ),
                    ],
                )],
                sliders=[dict(
                    active=len(frames) - 1,   # start at most recent
                    currentvalue=dict(
                        prefix="Date: ",
                        font=dict(size=13),
                        visible=True,
                    ),
                    pad=dict(t=55, b=10),
                    steps=[dict(
                        method="animate",
                        label=f.name,
                        args=[[f.name], dict(
                            mode="immediate",
                            frame=dict(duration=300, redraw=True),
                            transition=dict(duration=0),
                        )],
                    ) for f in frames],
                )],
            ),
        )
        fig.show()

        # ── Summary ─────────────────────────────────────────────────────────
        print(f"Animation: {len(frames)} weekly frames")
        print(f"  Date range : {frame_data[0][0].date()} → {frame_data[-1][0].date()}")
        print(f"  Y range    : [{y_range[0]:.2f}, {y_range[1]:.2f}] €/MWh (fixed)")
        if "M1" in ttf_df.columns:
            m1_now  = float(ttf_df.loc[init_dt, "M1"])
            m1_1yr  = float(ttf_df.loc[
                ttf_df.index.get_indexer(
                    [init_dt - pd.DateOffset(years=1)], method="nearest"
                )[0], "M1"
            ])
            print(f"  M1 today   : {m1_now:.2f} EUR/MWh")
            print(f"  M1 1yr ago : {m1_1yr:.2f} EUR/MWh  ({m1_now - m1_1yr:+.2f})")
